# Free-GPU ASR for 7-hour videos

Runs on **Kaggle** (T4/P100, 30 GPU h/week), **Google Colab** (T4), or **Lightning AI** (80 GPU h/month).

1. Set runtime to **GPU** (Kaggle: Settings → Accelerator → GPU T4; Colab: Runtime → Change runtime type → T4).
2. Put your video URL in the next cell.
3. Run all. Download `transcript.json` at the end.
4. In ClipForge: **Import transcript** and paste the same URL. Clips render from sections — no local transcription.

In [ ]:
URL = 'https://www.twitch.tv/videos/2934887631'  # <-- change this
MODEL = 'large-v3'   # large-v3 = best accuracy on T4; 'medium' = faster
print('URL:', URL)

In [ ]:
!pip -q install -U faster-whisper yt-dlp

In [ ]:
import subprocess, os, glob
print('GPU check:')
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
subprocess.run(['yt-dlp','-f','bestaudio/best','-x','--audio-format','wav',
                '--postprocessor-args','ffmpeg:-ar 16000 -ac 1',
                '-o','audio.%(ext)s', URL], check=True)
audio = glob.glob('audio*.wav')[0]
print('Audio:', audio, os.path.getsize(audio)//1024//1024, 'MB')

In [ ]:
from faster_whisper import WhisperModel
import time
t0 = time.time()
model = WhisperModel(MODEL, device='cuda', compute_type='float16')
segments, info = model.transcribe(audio, beam_size=5, word_timestamps=True, vad_filter=True)
out = []
for s in segments:
    words = [{'word': w.word, 'start': round(w.start,2), 'end': round(w.end,2)}
             for w in (s.words or []) if w.word]
    out.append({'start': round(s.start,2), 'end': round(s.end,2),
                'text': s.text.strip(), 'words': words})
print(f'Done in {(time.time()-t0)/60:.1f} min | {len(out)} segments | lang={info.language}')

In [ ]:
import json
result = {'language': info.language, 'duration': round(info.duration,2),
          'backend': f'free-gpu:{MODEL}', 'segments': out,
          'words': [w for s in out for w in s['words']]}
with open('transcript.json','w',encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False)
print('Saved transcript.json', os.path.getsize('transcript.json')//1024, 'KB')
from google.colab import files  # Kaggle: use files.download or skip
files.download('transcript.json')